# Track 03.2 — Advanced Reranking & Cross-Encoder Architectures

In dense vector retrieval (Track 02), we learned that **Bi-Encoders** independently map queries and documents into isolated vector representations $u = f_{\theta}(q)$ and $v = f_{\theta}(d)$. While this enables sub-millisecond approximate nearest neighbor (ANN) search over millions of documents via dot-products ($O(N \cdot D)$), it incurs a severe **Representation Bottleneck**: query tokens cannot attend to document tokens.

As a result, Bi-Encoders struggle with:
1. **Adversarial Lexical Overlap:** Documents sharing 90% of words with the query but asserting opposite facts (false positives).
2. **Negation & Polarity Inversion:** Queries specifying `"without Docker"` matching documents describing `"with Docker"`.
3. **Complex Conditionality:** Multi-constraint filtering and subtle predicate logic.

**Cross-Encoders** eliminate the representation bottleneck by concatenating query and document into a single joint sequence:
$$\text{Input} = [\text{CLS}] \circ q \circ [\text{SEP}] \circ d \circ [\text{SEP}]$$
Every query token attends directly to every document token across all $M$ transformer layers and attention heads via full cross-attention.

```mermaid
flowchart TD
    subgraph BiEncoder["Bi-Encoder Architecture (Fast / Candidate Generation)"]
        direction TB
        Q1["Query: 'ChromaDB without Docker'"] --> E1["Encoder f(q)"] --> V1["Vector u in R^D"]
        D1["Doc: 'ChromaDB with Docker'"] --> E2["Encoder f(d)"] --> V2["Vector v in R^D"]
        V1 & V2 --> Dot["Cosine Sim: <u, v> = O(N * D)"]
    end

    subgraph CrossEncoder["Cross-Encoder Architecture (Accurate / Joint Attention)"]
        direction TB
        QD["Concat: [CLS] Query [SEP] Doc [SEP]"] --> Joint["Full Cross-Attention Transformer
O(N * M * L^2 * D)"]
        Joint --> ClsHead["Classification Head (Logit z)"]
        ClsHead --> Sigmoid["Sigmoid Calibration P(Relevant | q, d)"]
    end

    classDef be fill:#eff6ff,stroke:#3b82f6,stroke-width:1.5px,color:#1e3a8a;
    classDef ce fill:#fef2f2,stroke:#ef4444,stroke-width:1.5px,color:#991b1b;
    class BiEncoder,Q1,E1,V1,D1,E2,V2,Dot be;
    class CrossEncoder,QD,Joint,ClsHead,Sigmoid ce;
```

### The Two-Stage Cascaded Retrieval Paradigm
Because full cross-attention scales quadratically with sequence length ($O(N \cdot M \cdot L^2)$), running a Cross-Encoder over an entire 1,000,000-document database would take minutes per query. Production systems resolve this using **Two-Stage Cascaded Retrieval**:
1. **Stage 1 (Coarse Candidate Retrieval):** Bi-Encoder / ANN retrieves candidate set $\mathcal{C}$ ($K_1 \approx 50\text{--}100$) in $<5\text{ ms}$.
2. **Stage 2 (Fine-Grained Precision Reranking):** Cross-Encoder jointly scores only the $K_1$ candidates in $\sim 20\text{ ms}$.
3. **Stage 3 (Calibration & Truncation):** Logits are calibrated via Sigmoid and truncated to top-$K_2$ ($K_2 \approx 3\text{--}5$) for the LLM prompt.

## 0. Environment Setup & Configuration

We configure the environment for headless execution, importing our shared utilities from `common.reranking` alongside NumPy and Matplotlib.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np

# Ensure clean headless plotting in isolated environments
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Ensure repository root is on sys.path
REPO_ROOT = Path(".").resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "tracks").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from common.reranking import (
    ScoredDocument,
    BiEncoderSimulator,
    CrossEncoderSimulator,
    ScoreCalibrator,
    CascadedRetrievalPipeline,
    RankingMetrics,
    GroqLLMReranker,
)

print(f"Environment initialized successfully. Repository root: {REPO_ROOT}")
print("100% offline CPU mode ready.")

## 1. Curated Hard-Negative Evaluation Corpus

To evaluate the architectural difference between Bi-Encoders and Cross-Encoders rigorously, we use a curated technical corpus containing subtle semantic distractors (hard negatives):
- Document `doc_chroma_with_docker` has dense lexical overlap with containerized deployment queries, but describes running *with* Docker.
- Document `doc_chroma_without_docker` directly answers how to run ChromaDB in Python *without* Docker.
- Other documents cover HNSW indexing, CUDA memory issues, and async event loops.

In [ ]:
corpus = {
    "doc_chroma_with_docker": (
        "How to run ChromaDB with Docker containers and docker-compose on production Linux servers "
        "with volume mounts and network isolation."
    ),
    "doc_chroma_without_docker": (
        "You can run ChromaDB directly as an in-process Python library without Docker using pip install chromadb "
        "and chromadb.PersistentClient()."
    ),
    "doc_hnsw_clustering": (
        "HNSW builds multi-layer proximity graphs for logarithmic approximate nearest neighbor vector search "
        "with high recall and low latency."
    ),
    "doc_cuda_oom": (
        "PyTorch CUDA out-of-memory errors occur when GPU tensor allocations exceed physical VRAM during training "
        "or large batch inference."
    ),
    "doc_async_event_loop": (
        "Python asyncio event loop coordinates non-blocking coroutines and task scheduling on a single OS thread."
    ),
}

# Challenging test query featuring polarity condition and adversarial lexical overlap:
query = "How to setup ChromaDB without Docker containers"
ground_truth_relevant = {"doc_chroma_without_docker"}
relevance_grades = {
    "doc_chroma_without_docker": 3.0,
    "doc_chroma_with_docker": 0.0,    # Severe distractor (contradicts intent)
    "doc_hnsw_clustering": 0.0,
    "doc_cuda_oom": 0.0,
    "doc_async_event_loop": 0.0,
}

print(f"Target Query: '{query}'")
print(f"Corpus size: {len(corpus)} documents.")

## 2. Stage 1: Bi-Encoder Candidate Generation & Representation Bottleneck Failure

Let's inspect how the Bi-Encoder scores this query against the corpus.

Because the query contains `"ChromaDB"`, `"Docker"`, and `"containers"`, and `doc_chroma_with_docker` contains all three words with high frequency, the Bi-Encoder assigns it a strong similarity score. The Bi-Encoder fails to recognize that `"without"` modifies `"Docker containers"`, incorrectly ranking the distractor as Rank 1 (False Positive).

In [ ]:
bi_encoder = BiEncoderSimulator(dim=64)
candidates = bi_encoder.retrieve_candidates(query, corpus, top_k=5)

print("--- Stage 1: Bi-Encoder Candidate Ranking ---")
for doc in candidates:
    print(f"Rank {doc.rank} | Cosine: {doc.bi_encoder_score:+.4f} | ID: {doc.doc_id}")
    print(f"   Excerpt: {doc.text[:80]}...")

bi_top1 = candidates[0].doc_id
print(f"\nBi-Encoder Top-1 Pick: '{bi_top1}' (Is Correct: {bi_top1 in ground_truth_relevant})")

## 3. Stage 2: Cross-Encoder Joint Attention & Negation Resolution

Now let's apply the **Cross-Encoder Simulator**. 

The Cross-Encoder takes the concatenated tokens of the query and candidate passage and computes the full token-by-token cross-attention matrix:
$$A_{ij} = \text{softmax}\left(\frac{q_i \cdot k_j}{\sqrt{d}}\right)$$

When computing the interaction, the Cross-Encoder detects the dependency between the negation token `"without"` and the entity `"docker"`, penalizing documents asserting `"with docker"` and rewarding the passage asserting `"without docker"`.

In [ ]:
cross_encoder = CrossEncoderSimulator()
reranked = cross_encoder.rerank(query, candidates)

print("--- Stage 2: Cross-Encoder Reranked Candidates ---")
for doc in reranked:
    print(f"Rank {doc.rank} | Logit: {doc.cross_encoder_logit:+.4f} | Bi-Enc: {doc.bi_encoder_score:+.4f} | ID: {doc.doc_id}")
    print(f"   Excerpt: {doc.text[:80]}...")

ce_top1 = reranked[0].doc_id
print(f"\nCross-Encoder Top-1 Pick: '{ce_top1}' (Is Correct: {ce_top1 in ground_truth_relevant})")

## 4. Visualizing the Cross-Attention Token Interaction Heatmap

To see exactly what happens inside the transformer, we extract and plot the cross-attention matrix between the query tokens and the target document tokens.

Notice how query tokens `"without"`, `"docker"`, and `"containers"` attend specifically to the corresponding tokens in the document, allowing the model to verify semantic agreement.

In [ ]:
target_doc = corpus["doc_chroma_without_docker"]
q_tokens, d_tokens, attn_matrix = cross_encoder.compute_attention_matrix(query, target_doc)

fig, ax = plt.subplots(figsize=(11, 4.5))
cax = ax.matshow(attn_matrix, cmap="magma", aspect="auto")

ax.set_xticks(range(len(d_tokens)))
ax.set_xticklabels(d_tokens, rotation=45, ha="left", fontsize=9)
ax.set_yticks(range(len(q_tokens)))
ax.set_yticklabels(q_tokens, fontsize=10)

ax.set_title("Cross-Attention Alignment Heatmap: [Query Tokens x Document Tokens]", pad=20, fontsize=12, fontweight="bold")
fig.colorbar(cax, ax=ax, fraction=0.046, pad=0.04, label="Attention Weight")
plt.tight_layout()

plot_dir = REPO_ROOT / "assets" / "diagrams"
plot_dir.mkdir(parents=True, exist_ok=True)
plot_path_attn = plot_dir / "track03_cross_attention_heatmap.png"
plt.savefig(plot_path_attn, dpi=200, bbox_inches="tight")
plt.close(fig)

print(f"Cross-attention heatmap saved successfully to: {plot_path_attn}")

## 5. Score Calibration & Top-K Truncation

Raw transformer cross-encoder logits $z \in (-\infty, +\infty)$ cannot be directly compared across different queries or models, nor can they be used as absolute confidence thresholds.

We evaluate two calibration strategies:
1. **Sigmoid Calibration with Temperature Scaling:**
   $$\sigma(z; T) = \frac{1}{1 + e^{-z/T}}$$
   Maps raw logits into $[0, 1]$, representing the estimated posterior probability $P(\text{Relevant} \mid q, d)$.
   - $T = 1.0$: Standard logistic posterior.
   - $T < 1.0$: Sharpened distribution (higher contrast between top and bottom).
   - $T > 1.0$: Softened distribution.
2. **Min-Max Normalization:**
   $$s_{\text{norm}}(d_i) = \frac{z_i - \min(Z)}{\max(Z) - \min(Z) + \epsilon}$$
   Scales scores relative to the current candidate pool.

Finally, we apply **Top-$K$ Truncation with Confidence Gating** ($	au = 0.35$): even if $K_2 = 3$, documents with calibrated relevance below $\tau$ are rejected to protect downstream LLM prompts against irrelevant distractors.

In [ ]:
print("=== Score Calibration Comparison ===")
print(f"{'Doc ID':<28} | {'Raw Logit':<10} | {'Sigmoid (T=1.0)':<16} | {'Sigmoid (T=0.5)':<16} | {'Min-Max':<10}")
print("-" * 90)

logits = [d.cross_encoder_logit for d in reranked]
min_max_scores = ScoreCalibrator.min_max_normalize(logits)

for doc, mm in zip(reranked, min_max_scores):
    sig_1 = ScoreCalibrator.sigmoid(doc.cross_encoder_logit, temperature=1.0)
    sig_sharp = ScoreCalibrator.sigmoid(doc.cross_encoder_logit, temperature=0.5)
    print(f"{doc.doc_id:<28} | {doc.cross_encoder_logit:>+9.4f} | {sig_1:>15.4f} | {sig_sharp:>15.4f} | {mm:>9.4f}")

# Calibrate in-place with Sigmoid
ScoreCalibrator.calibrate_sigmoid(reranked, temperature=1.0)

# Apply threshold truncation (K=3, tau=0.35)
gated_passages = ScoreCalibrator.truncate_with_threshold(reranked, top_k=3, relevance_threshold=0.35)
print(f"\n--- Gated Context Passages Passed to LLM Context Window ({len(gated_passages)} retained) ---")
for doc in gated_passages:
    print(f"Rank {doc.rank} | Calibrated Score: {doc.calibrated_score:.4f} | ID: {doc.doc_id}")

## 6. End-to-End Cascaded Pipeline & Quantitative Precision Gains

We now package this into a unified `CascadedRetrievalPipeline` and measure information retrieval metrics:
- **Precision@1 & Precision@3:** Fraction of top-$k$ retrieved documents that are relevant.
- **Recall@3:** Fraction of total relevant documents retrieved in top-3.
- **Reciprocal Rank (RR):** Multiplicative inverse of the rank of the first relevant document ($1 / \text{rank}$).
- **NDCG@3:** Graded relevance metric rewarding high-relevance passages placed at the top.

In [ ]:
pipeline = CascadedRetrievalPipeline()
stage1_cand, stage2_final = pipeline.execute(
    query=query,
    corpus=corpus,
    candidate_k=5,
    final_k=3,
    temperature=1.0,
    threshold=0.35,
)

stage1_ids = [d.doc_id for d in stage1_cand]
stage2_ids = [d.doc_id for d in stage2_final]

# Compute metrics
metrics_stage1 = {
    "Precision@1": RankingMetrics.precision_at_k(stage1_ids, ground_truth_relevant, k=1),
    "Precision@3": RankingMetrics.precision_at_k(stage1_ids, ground_truth_relevant, k=3),
    "Recall@3": RankingMetrics.recall_at_k(stage1_ids, ground_truth_relevant, k=3),
    "MRR": RankingMetrics.reciprocal_rank(stage1_ids, ground_truth_relevant),
    "NDCG@3": RankingMetrics.ndcg_at_k(stage1_ids, relevance_grades, k=3),
}

metrics_stage2 = {
    "Precision@1": RankingMetrics.precision_at_k(stage2_ids, ground_truth_relevant, k=1),
    "Precision@3": RankingMetrics.precision_at_k(stage2_ids, ground_truth_relevant, k=3),
    "Recall@3": RankingMetrics.recall_at_k(stage2_ids, ground_truth_relevant, k=3),
    "MRR": RankingMetrics.reciprocal_rank(stage2_ids, ground_truth_relevant),
    "NDCG@3": RankingMetrics.ndcg_at_k(stage2_ids, relevance_grades, k=3),
}

print("=== Quantitative Retrieval Benchmark: Bi-Encoder vs Cascaded Reranker ===")
print(f"{'Metric':<16} | {'Stage 1 (Bi-Encoder)':<22} | {'Stage 2 (Cross-Encoder)':<24} | {'Gain / Delta':<12}")
print("-" * 80)
for m_name in metrics_stage1:
    v1 = metrics_stage1[m_name]
    v2 = metrics_stage2[m_name]
    delta = v2 - v1
    print(f"{m_name:<16} | {v1:<22.4f} | {v2:<24.4f} | {delta:>+10.4f}")

## 7. Comparative Performance Visualization

Let's visualize the metric comparison as a bar chart, demonstrating how cross-encoder reranking recovers from bi-encoder precision failures on hard-negative queries.

In [ ]:
metric_names = list(metrics_stage1.keys())
values_stage1 = [metrics_stage1[m] for m in metric_names]
values_stage2 = [metrics_stage2[m] for m in metric_names]

x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4.5))
rects1 = ax.bar(x - width/2, values_stage1, width, label="Stage 1: Bi-Encoder (ANN)", color="#93c5fd")
rects2 = ax.bar(x + width/2, values_stage2, width, label="Stage 2: Cross-Encoder Reranker", color="#2563eb")

ax.set_ylabel("Score (0.0 to 1.0)", fontsize=11)
ax.set_title("Retrieval Quality Gains via Cross-Encoder Reranking", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(metric_names, fontsize=10)
ax.set_ylim(0.0, 1.15)
ax.legend(frameon=True, facecolor="white", loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)

# Attach values above bars
for rect in rects1:
    h = rect.get_height()
    ax.annotate(f"{h:.2f}", xy=(rect.get_x() + rect.get_width() / 2, h), xytext=(0, 3),
                textcoords="offset points", ha="center", va="bottom", fontsize=9)
for rect in rects2:
    h = rect.get_height()
    ax.annotate(f"{h:.2f}", xy=(rect.get_x() + rect.get_width() / 2, h), xytext=(0, 3),
                textcoords="offset points", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plot_path_bench = plot_dir / "track03_reranking_benchmark.png"
plt.savefig(plot_path_bench, dpi=200, bbox_inches="tight")
plt.close(fig)

print(f"Benchmark chart saved successfully to: {plot_path_bench}")

## 8. Production LLM-as-a-Reranker (Optional Groq Execution)

In production environments where dedicated cross-encoder inference servers (such as Triton or TEI) are not hosted, an LLM can act as an ad-hoc cross-attention reranker.

The cell below checks for `GROQ_API_KEY`. If present and configured, it executes a live LLM reranking call. Otherwise, it gracefully uses our offline deterministic simulator with zero network calls.

In [ ]:
groq_key = os.getenv("GROQ_API_KEY")
if groq_key:
    print("GROQ_API_KEY detected. Initializing Groq LLM-as-a-Reranker (llama-3.3-70b-versatile)...")
    llm_reranker = GroqLLMReranker(api_key=groq_key)
    live_results = llm_reranker.rerank(query, candidates, top_k=3)
    print("\n--- Live Groq LLM Reranking Results ---")
    for doc in live_results:
        print(f"Rank {doc.rank} | Calibrated Score: {doc.calibrated_score:.2f} | ID: {doc.doc_id}")
else:
    print("No GROQ_API_KEY detected. Running in verified offline deterministic fallback mode.")
    offline_reranker = GroqLLMReranker(api_key=None)
    offline_results = offline_reranker.rerank(query, candidates, top_k=3)
    print("\n--- Verified Offline Reranker Fallback ---")
    for doc in offline_results:
        print(f"Rank {doc.rank} | Calibrated Score: {doc.calibrated_score:.2f} | ID: {doc.doc_id}")

## 9. Production Architecture & Latency-Cost Engineering Trade-offs

| Architectural Dimension | Bi-Encoder (Dense ANN) | Cross-Encoder (Reranker) | ColBERT / Late Interaction | LLM-as-Reranker |
|---|---|---|---|---|
| **Computational Complexity** | $O(N \cdot D)$ | $O(K_1 \cdot M \cdot L^2 \cdot D)$ | $O(K_1 \cdot L_q \cdot L_d \cdot D)$ | $O(K_1 \cdot \text{Tokens} \cdot \text{LLM-Layers})$ |
| **Indexing Requirement** | Pre-compute & index vectors offline | No offline document indexing (raw text) | Pre-compute token multi-vectors | No offline indexing |
| **Typical Latency** | $1\text{--}5\text{ ms}$ (1M docs) | $15\text{--}50\text{ ms}$ (50 docs) | $5\text{--}15\text{ ms}$ (500 docs) | $200\text{--}800\text{ ms}$ (20 docs) |
| **Storage Overhead** | Moderate ($D \times 4$ bytes/doc) | Zero vector storage | Very High (100+ vectors/doc) | Zero vector storage |
| **Negation & Polysemy Handling** | Poor (Bottleneck loss) | State-of-the-art | High | State-of-the-art |
| **Production Role** | Stage 1: Coarse retrieval | Stage 2: Precision reranking | Alternative hybrid stage | Low-volume / complex QA |

### Recommended Production Rerankers:
1. **CPU / Edge-friendly:** `cross-encoder/ms-marco-MiniLM-L-6-v2` (22M params, $\sim 15\text{ ms}$ on CPU).
2. **High-Accuracy Server:** `BAAI/bge-reranker-large` (560M params) or `BAAI/bge-reranker-v2-m3` (multilingual).
3. **Hosted Commercial API:** Cohere Rerank v3.5 (`rerank-english-v3.0`).

### Golden Rules for Reranking in RAG:
1. **Size Candidate Pool Appropriately:** Set $K_1 \in [30, 100]$. Reranking more than 100 documents yields diminishing recall returns while multiplying GPU/CPU latency linearly.
2. **Always Apply Score Calibration:** Convert raw logits to calibrated probabilities via Sigmoid. Logits are model-dependent and fluctuate across model updates.
3. **Guard Prompt Context with Thresholding:** Truncate to top-$K_2$ ($3\text{--}5$) and drop documents below a relevance threshold $\tau \approx 0.35$. Including low-confidence documents increases generator hallucination risk.